In [ ]:
# IMPORTS
from memo import memo, domain
from math import exp
import jax
import numpy as np
import jax.numpy as jnp
from jax.scipy.stats import beta as jax_beta
import matplotlib.pyplot as plt

# GLOBALS
INFO, ACTION = 0, 1
CAN, IMP, NULL = 0, 1, 2
goals = [INFO, ACTION]
U = [CAN, IMP, NULL]

# PARAMETERS
delta = 0.7      # face-threat cost (Brown & Levinson): social risk of IMP
prior_ga = .5    # uniform prior over goals

f_vals = np.linspace(0.01, 0.99, 50)     # grid over feasibility
theta_vals = np.linspace(0.01, 0.99, 20) # threshold values
kappa = 20   # tightness of beta_agg dist (a+b)
f_k = 10     # concentration of feasibility prior
alpha = 10   # softmax rationality param

# CONTEXTS: single parameter f_ctx = mu_circ(VP)
# f_ctx encodes the circumstantial feasibility of the VP.
# Low f_ctx → genuinely uncertain → CAN is informative as a question
# High f_ctx → obviously feasible → CAN is uninformative → indirect request
contexts = {
    'benchpress 150': 0.10,
    'drive me home':  0.70,
    'call me later':  0.90,
    'pass the salt':  0.99,
}

# FEASIBILITY PRIOR
# Beta prior over f, concentrated around f_ctx with tightness f_k.
def f_prior(f, f_ctx):
    f_c = jnp.clip(f, .01, .99)
    return jax_beta.pdf(f_c, f_ctx * f_k, (1 - f_ctx) * f_k)

In [ ]:
## A0 -- literal actioner
# CAN and IMP share the same truth-conditional core (both depend on ability)
# but have different speech act types:
#   CAN = question about ability → elicits an ANSWER (resolves INFO QUD)
#   IMP = directive to act       → elicits COMPLIANCE (satisfies ACTION goal)
#
# Acting entails answering (doing X proves you can), but not vice versa.

# P(f_agg > theta | speaker belief = f)
def p_yes(f, t):
    f_c = jnp.clip(f, .01, .99)
    return 1.0 - jax_beta.cdf(t, f_c * kappa, (1 - f_c) * kappa)

@jax.jit
def A0_answer(u, f, t):
    """Prob of answering 'yes' — resolves INFO QUD.
    Only CAN (a question) elicits a literal answer."""
    p = p_yes(f, t)
    return jnp.where(u == CAN, p, 0.)

@jax.jit
def A0_comply(u, f, t):
    """Prob of taking the action — satisfies ACTION goal.
    Only IMP (a directive) elicits literal compliance.
    (CAN's indirect-request reading emerges at S2 via goal inference.)"""
    p = p_yes(f, t)
    return jnp.where(u == IMP, p, 0.)

print(f"A0_answer(can, ...) = {float(A0_answer(CAN, f_vals[25], theta_vals[1])):.3f}")
print(f"A0_answer(imp, ...) = {float(A0_answer(IMP, f_vals[25], theta_vals[1])):.3f}")
print(f"A0_comply(can, ...) = {float(A0_comply(CAN, f_vals[25], theta_vals[1])):.3f}")
print(f"A0_comply(imp, ...) = {float(A0_comply(IMP, f_vals[25], theta_vals[1])):.3f}")

In [ ]:
## S1
u_label = {CAN: 'can', IMP: 'imp', NULL: 'null'}
goal_label = {INFO: 'info', ACTION: 'action'}

# INFO UTILITY: expected information gain from A0's yes/no answer
# IG(f_ctx, θ) = H[f] − P(yes|θ)·H[f|yes,θ] − P(no|θ)·H[f|no,θ]
# Marginalizes over the speaker's prior π(f) to compute expected
# uncertainty reduction about f from A0's binary response.
@jax.jit
def discrete_entropy(probs):
    p_c = jnp.clip(probs, 1e-12, 1.0)
    return -jnp.sum(p_c * jnp.log(p_c))

@jax.jit
def info_gain(f_ctx, t):
    f_arr = jnp.array(f_vals)
    raw = jax_beta.pdf(jnp.clip(f_arr, .01, .99), f_ctx * f_k, (1 - f_ctx) * f_k)
    pi_f = raw / jnp.sum(raw)
    p_y = jax.vmap(lambda f: p_yes(f, t))(f_arr)
    p_yes_marg = jnp.sum(pi_f * p_y)
    p_no_marg = 1.0 - p_yes_marg
    post_yes = pi_f * p_y / jnp.clip(p_yes_marg, 1e-12)
    post_no = pi_f * (1 - p_y) / jnp.clip(p_no_marg, 1e-12)
    h_prior = discrete_entropy(pi_f)
    h_yes = discrete_entropy(post_yes)
    h_no = discrete_entropy(post_no)
    return h_prior - p_yes_marg * h_yes - p_no_marg * h_no

@jax.jit
def eu_s1(u, g, f, t, f_ctx):
    act = jnp.where(u == IMP, p_yes(f, t), 0.)
    ig = info_gain(f_ctx, t)
    v = jnp.where(u == CAN, ig, 0.)        # INFO value from CAN
    social = jnp.where(u == IMP, delta, 0.) # face-threat cost of IMP
    return jnp.where(g == ACTION, act, v) - social

# P(u | g, f, t)
@memo
def S1[_u: U, _g: goals, _f: f_vals, _t: theta_vals](f_ctx, alpha):
    speaker: knows(_g, _f, _t)
    speaker: chooses(u in U, wpp=exp(alpha * eu_s1(u, _g, _f, _t, f_ctx)))
    return Pr[speaker.u == _u]

s1_out = S1(f_ctx=0.5, alpha=alpha)
ti_mid = int(len(theta_vals) // 2)
for gi, g in enumerate(goals):
    probs = {u_label[u]: round(float(s1_out[ui, gi, 25, ti_mid]), 3) for ui, u in enumerate(U)}
    print(f"S1(g={goal_label[g]}, f=.5, t={theta_vals[ti_mid]:.2f}) = {probs}")

In [ ]:
## A1

@jax.jit
def goal_prior(g, prior_ga):
    return jnp.where(g == 1, prior_ga, 1 - prior_ga)

@memo
def A1[_g: goals](f_ctx, prior_ga, alpha):
    a1: knows(_g)
    a1: thinks[
        spk: given(g in goals, wpp=goal_prior(g, prior_ga)),
        spk: given(f in f_vals, wpp=f_prior(f, f_ctx)),
        spk: given(t in theta_vals, wpp=1),
        spk: chooses(u in U, wpp=S1[u, g, f, t](f_ctx, alpha))
    ]
    a1: observes_that[spk.u == 0]
    return a1[Pr[spk.g == _g]]

def a1_act(f_ctx, prior_ga=0.5):
    p_ga = float(A1(f_ctx=f_ctx, prior_ga=prior_ga, alpha=alpha)[ACTION])
    eu_do = f_ctx
    eu_ans = 1 - p_ga
    probs = jnp.exp(alpha * jnp.array([eu_do, eu_ans]))
    return float(probs[0] / probs.sum())

print("A1 goal inference by context:")
for label, f_ctx in contexts.items():
    p_ga = float(A1(f_ctx=f_ctx, prior_ga=0.5, alpha=alpha)[ACTION])
    print(f"  {label:<18} f_ctx={f_ctx:.2f}: P(g=action|u=can) = {p_ga:.3f}, P(act) = {a1_act(f_ctx):.3f}")

In [ ]:
## S2
# S2 knows A1 does goal inference, so CAN now has action utility:
# A1 may infer g=ACTION from CAN and comply → p_a1_do

@jax.jit
def eu_s2(u, g, f, t, f_ctx, p_a1_do):
    act = jnp.where(u == IMP, p_yes(f, t), 0.)
    ig = info_gain(f_ctx, t)
    v = jnp.where(u == CAN, ig, 0.)
    # CAN gets indirect action value via A1's goal inference
    act_s2 = jnp.where(u == CAN, p_a1_do, act)
    social = jnp.where(u == IMP, delta, 0.)
    return jnp.where(g == ACTION, act_s2, v) - social

@memo
def S2[_u: U, _g: goals, _f: f_vals, _t: theta_vals](f_ctx, p_a1_do, alpha):
    speaker: knows(_g, _f, _t)
    speaker: chooses(u in U, wpp=exp(alpha * eu_s2(u, _g, _f, _t, f_ctx, p_a1_do)))
    return Pr[speaker.u == _u]

s2_out = S2(f_ctx=0.5, p_a1_do=a1_act(0.5), alpha=alpha)
for gi, g in enumerate(goals):
    probs = {u_label[u]: round(float(s2_out[ui, gi, 25, ti_mid]), 3) for ui, u in enumerate(U)}
    print(f"S2(g={goal_label[g]}, f=.5, t={theta_vals[ti_mid]:.2f}) = {probs}")

In [ ]:
## A2

@memo
def A2[_g: goals](f_ctx, prior_ga, p_a1_do, alpha):
    a2: knows(_g)
    a2: thinks[
        spk: given(g in goals, wpp=goal_prior(g, prior_ga)),
        spk: given(f in f_vals, wpp=f_prior(f, f_ctx)),
        spk: given(t in theta_vals, wpp=1),
        spk: chooses(u in U, wpp=S2[u, g, f, t](f_ctx, p_a1_do, alpha))
    ]
    a2: observes_that[spk.u == 0]
    return a2[Pr[spk.g == _g]]

def a2_act(f_ctx, prior_ga=0.5):
    p_a1 = a1_act(f_ctx, prior_ga)
    p_ga = float(A2(f_ctx=f_ctx, prior_ga=prior_ga, p_a1_do=p_a1, alpha=alpha)[ACTION])
    eu_do = f_ctx
    eu_ans = 1 - p_ga
    probs = jnp.exp(alpha * jnp.array([eu_do, eu_ans]))
    return float(probs[0] / probs.sum())

print("A2 goal inference by context:")
for label, f_ctx in contexts.items():
    p_a1 = a1_act(f_ctx)
    p_ga = float(A2(f_ctx=f_ctx, prior_ga=0.5, p_a1_do=p_a1, alpha=alpha)[ACTION])
    print(f"  {label:<18} f_ctx={f_ctx:.2f}: P(g=action|u=can) = {p_ga:.3f}, P(act) = {a2_act(f_ctx):.3f}")

In [ ]:
## plots: A1 vs A2

def act_at(f_ctx, prior_ga=0.5):
    """P(do X) for A1 and A2 at given feasibility and prior"""
    eu_do = f_ctx

    p_ga1   = float(A1(f_ctx=f_ctx, prior_ga=prior_ga, alpha=alpha)[ACTION])
    eu_ans1 = 1 - p_ga1
    p1      = jnp.exp(alpha * jnp.array([eu_do, eu_ans1]))
    p_a1    = float(p1[0] / p1.sum())

    p_ga2   = float(A2(f_ctx=f_ctx, prior_ga=prior_ga, p_a1_do=p_a1, alpha=alpha)[ACTION])
    eu_ans2 = 1 - p_ga2
    p2      = jnp.exp(alpha * jnp.array([eu_do, eu_ans2]))
    p_a2    = float(p2[0] / p2.sum())

    return p_a1, p_a2

priors = np.linspace(0.01, 0.99, 60)
f_sweep = np.linspace(0.05, 0.99, 60)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
a1v, a2v = zip(*[act_at(0.5, p) for p in priors])
ax.plot(priors, a1v, label='A1', color='steelblue')
ax.plot(priors, a2v, label='A2', color='tomato')
ax.axvline(prior_ga, color='gray', linestyle='--', linewidth=.8, label=f'current prior={prior_ga}')
ax.set_xlabel('prior P(g_action)')
ax.set_ylabel('P(act | u_can)')
ax.set_title('vs prior (f_ctx=0.5)')
ax.legend(); ax.set_ylim(0, 1)

ax = axes[1]
a1v, a2v = zip(*[act_at(f) for f in f_sweep])
ax.plot(f_sweep, a1v, label='A1', color='steelblue')
ax.plot(f_sweep, a2v, label='A2', color='tomato')
ax.set_xlabel('f_ctx (circumstantial feasibility)')
ax.set_ylabel('P(act | u_can)')
ax.set_title('vs feasibility')
ax.legend(); ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
## plots: S1 vs S2
s2_out_curr = S2(f_ctx=0.5, p_a1_do=a1_act(0.5), alpha=alpha)

u_colors = {CAN: 'steelblue', IMP: 'tomato', NULL: 'gray'}
u_labels = {CAN: 'can', IMP: 'imp', NULL: 'null'}

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)

for ri, (gi, glabel) in enumerate([(ACTION, 'g=action'), (INFO, 'g=info')]):
    for ci, (out, mlabel) in enumerate([(s1_out, 'S1'), (s2_out_curr, 'S2')]):
        ax = axes[ri, ci]
        for ui in U:
            ax.plot(f_vals, out[ui, gi, :, ti_mid],
                    color=u_colors[ui], label=u_labels[ui], linewidth=2)
        ax.set_title(f'{mlabel} | {glabel}')
        ax.set_ylim(0, 1)
        if ri == 1: ax.set_xlabel('f (feasibility)')
        if ci == 0: ax.set_ylabel('P(utterance | g, f)')
        ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
## Context comparison
# When ability is uncertain (benchpress), INFO value of CAN is high → literal question
# When ability is obvious (salt), INFO value → 0 → CAN only useful via ACTION at S2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: goal inference by context
ax = axes[0]
x = np.arange(len(contexts))
width = 0.35
a1_goals = [float(A1(f_ctx=f, prior_ga=0.5, alpha=alpha)[ACTION])
             for f in contexts.values()]
a2_goals = [float(A2(f_ctx=f, prior_ga=0.5, p_a1_do=a1_act(f), alpha=alpha)[ACTION])
             for f in contexts.values()]
ax.bar(x - width/2, a1_goals, width, label='A1', color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.bar(x + width/2, a2_goals, width, label='A2', color='tomato', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(contexts.keys(), fontsize=9)
ax.set_ylabel('P(g=action | u=can)')
ax.set_title('Goal inference by context')
ax.legend(); ax.set_ylim(0, 1)

# Panel 2: compliance by context
ax = axes[1]
a1_acts = [a1_act(f) for f in contexts.values()]
a2_acts = [a2_act(f) for f in contexts.values()]
ax.bar(x - width/2, a1_acts, width, label='A1', color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.bar(x + width/2, a2_acts, width, label='A2', color='tomato', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(contexts.keys(), fontsize=9)
ax.set_ylabel('P(act | u=can)')
ax.set_title('Compliance by context')
ax.legend(); ax.set_ylim(0, 1)

# Panel 3: sweep f_ctx, show goal inference at A1 and A2
ax = axes[2]
f_ctx_sweep = np.linspace(0.05, 0.99, 30)
a1_sweep = []
a2_sweep = []
for f in f_ctx_sweep:
    g1 = float(A1(f_ctx=f, prior_ga=0.5, alpha=alpha)[ACTION])
    p_a1 = a1_act(f)
    g2 = float(A2(f_ctx=f, prior_ga=0.5, p_a1_do=p_a1, alpha=alpha)[ACTION])
    a1_sweep.append(g1)
    a2_sweep.append(g2)
ax.plot(f_ctx_sweep, a1_sweep, label='A1', color='steelblue', linewidth=2)
ax.plot(f_ctx_sweep, a2_sweep, label='A2', color='tomato', linewidth=2)
ax.set_xlabel('f_ctx (circumstantial feasibility)')
ax.set_ylabel('P(g=action | u=can)')
ax.set_title('Goal inference vs feasibility')
ax.legend(); ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()